# 00 — Context: Decoherence-Free Sensing

**Repo:** `decoherence-free-sensing`  
**Lab report:** *Decoherence-Free Subspaces Specify Robust Quantum Sensing*  
**Paper:** Kaubruegger *et al.*, *Physical Review X* **16**, 021052 (2026), DOI: `10.1103/ksyh-mb4s`

This notebook establishes the context for the repo:

\[
\text{common-mode noise}
\;\longrightarrow\;
\text{DFS constraint}
\;\longrightarrow\;
\text{DFS entanglement}
\;\longrightarrow\;
\text{robust differential phase sensing}.
\]

## Engineering statement

The leading specification is **not entanglement**.

The leading specification is **rejection of common-mode noise**.

Entanglement is engineered only after the admissible subspace has been specified.

## Notebook outputs

Running this notebook creates:

- `results/figures/00_leading_specification_flow_v2.png`
- `results/figures/13_phase_coordinates_v2.png`
- `results/figures/17_common_mode_noise_cloud_v2.png`
- `results/figures/17_common_mode_histogram_v2.png`
- `results/figures/23_dfs_vs_full_hilbert_scaling_v2.png`
- `results/figures/29_sql_hl_lieb_mattis_scaling_v2.png`
- `results/figures/37_two_body_fringe_response_v2.png`
- `results/figures/43_preparation_protocol_map_v2.png`
- `results/context_summary.json`
- `results/context_summary.csv`
- `results/decoherence_free_sensing_context_outputs.zip`

In [ ]:
from pathlib import Path
import json
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})

## 1. Leading specification flow

The design rule is constraint-first:

1. identify the dominant shared error,
2. restrict to a DFS sector where that shared error acts trivially,
3. engineer entanglement inside the admissible sector,
4. estimate only the differential phase.

In [ ]:
def add_box(ax, xy, width, height, title, body):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.05",
        linewidth=1.8,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + width/2, y + height*0.62, title,
            ha="center", va="center", fontsize=15, fontweight="bold")
    ax.text(x + width/2, y + height*0.32, body,
            ha="center", va="center", fontsize=11)

def add_arrow(ax, x0, y0, x1, y1):
    arr = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    )
    ax.add_patch(arr)

fig, ax = plt.subplots(figsize=(13.5, 5.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.91, "Leading Specification Flow",
        ha="center", va="center", fontsize=24)

boxes = [
    (0.04, "Common-mode\nnoise", "same phase error\nat both nodes"),
    (0.285, "DFS constraint", "fixed $J_z^+$ sector\nshared phase becomes global"),
    (0.53, "DFS entanglement", "Lieb-Mattis state\nor preparation proxy"),
    (0.775, "Differential\nphase sensing", "signal survives;\nnoise rejected"),
]
w, h, y = 0.205, 0.30, 0.40
for x, title, body in boxes:
    add_box(ax, (x, y), w, h, title, body)

for x in [0.245, 0.49, 0.735]:
    add_arrow(ax, x, y + h/2, x + 0.035, y + h/2)

ax.text(
    0.5, 0.22,
    "Constraint first: remove the common phase $\\Phi$, then optimize sensitivity to the differential phase $\\varphi$.",
    ha="center", va="center", fontsize=15
)

path = FIGURES / "00_leading_specification_flow_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 2. Common and differential phase coordinates

The two-node phases are more naturally read in rotated coordinates:

\[
\Phi = \frac{\phi_A+\phi_B}{2},
\qquad
\varphi = \frac{\phi_A-\phi_B}{2}.
\]

Common-mode noise moves the system along the shared-noise direction. Differential sensing uses the separation between nodes.

In [ ]:
phi = np.linspace(-np.pi, np.pi, 300)

fig, ax = plt.subplots(figsize=(8.0, 6.8))

# Constant common phase Phi = c: phiB = 2c - phiA
# Constant differential phase varphi = c: phiB = phiA - 2c
constants = np.linspace(-np.pi, np.pi, 7)

for c in constants:
    ax.plot(phi, 2*c - phi, linewidth=1.0, alpha=0.55)
for c in constants:
    ax.plot(phi, phi - 2*c, linestyle="--", linewidth=1.0, alpha=0.55)

ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-np.pi, np.pi)
ax.set_xlabel(r"node A phase $\phi_A$")
ax.set_ylabel(r"node B phase $\phi_B$")
ax.set_title("Common and Differential Phase Coordinates")
ax.text(
    -2.9, 2.30,
    "solid: constant common phase $\\Phi$\n"
    "dashed: constant differential phase $\\varphi$",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.85)
)
ax.grid(True, alpha=0.3)

path = FIGURES / "13_phase_coordinates_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 3. Shared-noise cloud

A simulated pair of node observations makes the geometry visible:

\[
\phi_A^{obs} = \Phi_{noise} + \varphi + \epsilon_A,
\qquad
\phi_B^{obs} = \Phi_{noise} - \varphi + \epsilon_B.
\]

The common noise spreads both nodes along the diagonal. The differential estimate stays concentrated near the true phase.

In [ ]:
rng = np.random.default_rng(42)

nshots = 900
true_varphi = 0.23
common_noise = rng.normal(0.0, 1.35, nshots)
local_a = rng.normal(0.0, 0.07, nshots)
local_b = rng.normal(0.0, 0.07, nshots)

obs_a = common_noise + true_varphi + local_a
obs_b = common_noise - true_varphi + local_b

differential_estimate = (obs_a - obs_b) / 2
common_estimate = (obs_a + obs_b) / 2

fig, ax = plt.subplots(figsize=(8.0, 6.2))
ax.scatter(obs_a, obs_b, s=14, alpha=0.35)
ax.set_title("Common-Mode Noise Moves Both Nodes Together")
ax.set_xlabel(r"observed node A phase")
ax.set_ylabel(r"observed node B phase")
ax.grid(True, alpha=0.3)
ax.text(
    -3.35, 3.08,
    "points move along shared-noise diagonal\n"
    "separation still estimates differential phase",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.85)
)

path = FIGURES / "17_common_mode_noise_cloud_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.8))
ax.hist(common_estimate, bins=40, alpha=0.60, label="common estimate")
ax.hist(differential_estimate, bins=40, alpha=0.85, label="differential estimate")
ax.axvline(true_varphi, linestyle="--", linewidth=1.6, label="true differential phase")
ax.set_title("Differential Estimate Rejects Shared Noise")
ax.set_xlabel("phase estimate")
ax.set_ylabel("count")
ax.legend()
ax.grid(True, alpha=0.25)

path = FIGURES / "17_common_mode_histogram_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 4. DFS basis dimension

The full two-node Hilbert space grows exponentially with atom number. A permutation-symmetric central DFS sector grows only linearly.

This does not remove the physics. It isolates the admissible sector where common-mode phase noise is rejected.

In [ ]:
N = np.arange(2, 82, 2)
full_hilbert = 2.0 ** N
central_dfs_symmetric = N / 2 + 1

fig, ax = plt.subplots(figsize=(8.0, 6.0))
ax.semilogy(N, full_hilbert, linewidth=2.0, label=r"full Hilbert space $2^N$")
ax.semilogy(N, central_dfs_symmetric, linewidth=2.0, marker="o", markersize=3,
            label=r"symmetric central DFS basis $N/2+1$")
ax.set_title("DFS Constraint Turns an Exponential Search into a Linear Sector")
ax.set_xlabel("total atom number N")
ax.set_ylabel("basis size")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

path = FIGURES / "23_dfs_vs_full_hilbert_scaling_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 5. Scaling references

For unentangled atoms, the phase variance reference is the Standard Quantum Limit:

\[
\Delta^2_\varphi \sim \frac{1}{N}.
\]

The Heisenberg reference is

\[
\Delta^2_\varphi \sim \frac{1}{N^2}.
\]

For the Lieb-Mattis target state in the paper,

\[
F_Q=\frac{N^2+4N}{3},
\qquad
\Delta^2_\varphi \geq \frac{3}{N^2+4N}.
\]

In [ ]:
N = np.logspace(np.log10(4), np.log10(2048), 300)
sql = 1 / N
heisenberg = 1 / (N**2)
lieb_mattis_qcrb = 3 / (N**2 + 4*N)

fig, ax = plt.subplots(figsize=(8.2, 6.1))
ax.loglog(N, sql, linewidth=2.0, label="SQL: 1/N")
ax.loglog(N, heisenberg, linewidth=2.0, label="Heisenberg: 1/N²")
ax.loglog(N, lieb_mattis_qcrb, linewidth=2.0, label="Lieb-Mattis QCRB: 3/(N²+4N)")
ax.set_title("Scaling References for Differential Quantum Sensing")
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"variance reference $\Delta^2_\varphi$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

path = FIGURES / "29_sql_hl_lieb_mattis_scaling_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 6. DFS-compatible two-body fringe response

Inside the DFS, single-body coherences vanish. The paper therefore uses a compatible two-body observable:

\[
M = J_A^+J_B^- + J_A^-J_B^+.
\]

The normalized response has the form of a two-body fringe. The zero crossing at \(\varphi=\pi/4\) is the useful operating point because the slope is largest there.

In [ ]:
varphi = np.linspace(0, np.pi/2, 600)

fig, ax = plt.subplots(figsize=(8.2, 5.8))

for total_N in [40, 120, 320]:
    amplitude = (total_N**2 + 4*total_N) / 12
    response = -amplitude * np.cos(2 * varphi)
    response_normalized = response / amplitude
    ax.plot(varphi, response_normalized, linewidth=2.0, label=f"N={total_N}")

ax.axvline(np.pi/4, linestyle="--", linewidth=1.8, label=r"optimal operating point $\pi/4$")
ax.scatter([np.pi/4], [0], s=55, zorder=5)
ax.annotate(
    "max slope",
    xy=(np.pi/4, 0),
    xytext=(np.pi/4 + 0.12, -0.38),
    arrowprops=dict(arrowstyle="->", linewidth=1.2),
    fontsize=12
)

ax.set_title("DFS-Compatible Two-Body Fringe Response")
ax.set_xlabel(r"differential phase $\varphi$")
ax.set_ylabel(r"normalized $\langle M_\varphi\rangle$")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

path = FIGURES / "37_two_body_fringe_response_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 7. Preparation protocol map

The paper gives multiple routes into useful DFS-compatible entanglement:

- **Unitary** route: two-mode-squeezing quench proxy.
- **Dissipative** route: collective emission into the cavity.
- **Measurement-conditioned** route: use photon counts to decide whether to keep or restart a prepared state.

All routes are judged by the same specification: stay robust to shared noise while preserving differential sensitivity.

In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Preparation Protocol Map",
        ha="center", va="center", fontsize=22)

protocols = [
    (0.06, "Unitary", "two-mode squeezing\nquench proxy\n\nfast entangling\ndynamics"),
    (0.37, "Dissipative", "collective emission\ninto cavity\n\nrobust mixed\nDFS state"),
    (0.68, "Measurement-\nconditioned", "photon count\nselection\n\nrestart if too\nmany photons"),
]
w, h, y = 0.26, 0.36, 0.39
for x, title, body in protocols:
    add_box(ax, (x, y), w, h, title, body)

add_arrow(ax, 0.32, y + h/2, 0.37, y + h/2)
add_arrow(ax, 0.63, y + h/2, 0.68, y + h/2)

ax.text(
    0.5, 0.25,
    "All routes are judged by the same specification: stay robust to shared noise while preserving differential sensitivity.",
    ha="center", va="center", fontsize=14
)

path = FIGURES / "43_preparation_protocol_map_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 8. Context summary table

In [ ]:
summary = pd.DataFrame([
    {
        "item": "leading_specification",
        "value": "Decoherence-free subspaces reject common-mode phase noise before entanglement is optimized."
    },
    {
        "item": "common_phase",
        "value": "Phi = (phi_A + phi_B) / 2"
    },
    {
        "item": "differential_phase",
        "value": "varphi = (phi_A - phi_B) / 2"
    },
    {
        "item": "dfs_operator",
        "value": "J_z^+ = J_z^A + J_z^B"
    },
    {
        "item": "differential_generator",
        "value": "J_z^- = J_z^A - J_z^B"
    },
    {
        "item": "target_state_family",
        "value": "DFS entangled states; Lieb-Mattis target state as central example"
    },
    {
        "item": "target_qfi",
        "value": "F_Q = (N^2 + 4N) / 3"
    },
    {
        "item": "qcrb_reference",
        "value": "Delta_phi^2 >= 3 / (N^2 + 4N)"
    },
    {
        "item": "measurement",
        "value": "M = J_A^+ J_B^- + J_A^- J_B^+"
    },
    {
        "item": "repo_claim",
        "value": "Computational specifications for decoherence-free differential quantum sensing."
    },
])

summary_csv = RESULTS / "context_summary.csv"
summary_json = RESULTS / "context_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary

## 9. Export notebook outputs

This cell creates one zip archive containing all generated figures and context summary files.

In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_context_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(FIGURES.glob("*_v2.png")):
        zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created archive: {zip_path}")

# Colab download support; harmless in local Jupyter/GitHub Codespaces.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download manually from:", zip_path)

zip_path


## 10. Next notebook

Suggested next notebook:

`07_dfs_constraint.ipynb`

Core goal:

- explicitly construct \(J_z^+\) sectors,
- demonstrate that a common phase becomes a global phase inside one sector,
- show why DFS-compatible measurements must commute with \(J_z^+\).